# G1 single-unit output ablation (29 output units): screening, cross-seed sweep and graded attenuation

Notebook for: Obaru, K. "A Single Critical Output Unit in a Feedforward Bipedal Locomotion Policy: Cross-Seed Replication and Mechanism Analysis in the Unitree G1 Humanoid" (Frontiers in Robotics and AI, Manuscript ID 1957291).

このノートブックは、Unitree G1の学習済み歩行方策(純粋なMLP、再帰構造なし)について、出力層の各ユニットを1つずつ除去し、まず29関節を広くスクリーニングしてから、重要な関節だけを詳しく調べるものです。

構成(論文との対応):
- §1–3 セットアップ、出力層の構造の確認、除去関数の定義(論文§2.3)
- §4–5 3シードのスクリーニング(補足表S1)
- §6–8 4関節×9速度×8方策のスイープと左右比較(論文§3.2、図2、表2)
- §13 スクリーニングの8シード拡張(論文§3.1、表1の0.5 m/s列、図1)
- §16 段階的な出力減衰(論文§3.4、図4)
- §10–12, §14–15 旧版の機序解析(撤回済み。各節冒頭の注記を参照)


In [ ]:
#mount
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/content/drive/MyDrive/g1_cross_seed_logs/logs" /content/ && echo "復元しました" || echo "復元に失敗しました。パスを確認してください"

Mounted at /content/drive
復元しました


In [ ]:
import os
print("logsフォルダが存在するか:", os.path.exists("logs"))
if os.path.exists("logs/rsl_rl/g1_velocity"):
    print("中身:", os.listdir("logs/rsl_rl/g1_velocity"))

logsフォルダが存在するか: True
中身: ['2026-07-20_16-57-27_seed3', '2026-07-20_14-46-30_seed2', '2026-07-20_01-08-28_seed1', '2026-07-20_01-50-54_seed3', '2026-07-20_02-11-51_seed4', '2026-07-21_19-17-45_seed8', '2026-07-20_21-21-30_seed5', '2026-07-20_12-36-23_seed1', '2026-07-20_12-09-08_seed2', '2026-07-21_14-56-27_seed6', '2026-07-20_02-32-52_seed5', '2026-07-20_07-27-58_seed1', '2026-07-20_11-25-33_seed1', '2026-07-20_11-03-36_seed1', '2026-07-20_19-09-05_seed4', '2026-07-21_17-06-40_seed7', '2026-07-20_01-30-11_seed2']


## 1. セットアップ

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas

import torch
assert torch.cuda.is_available(), "GPU ランタイムに切り替えてください"
print("✅ install done:", torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.5/559.5 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 144.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## 2. 関節名の確認、および actor内部構造へのアクセス方法の検証(最重要)

§2-1で関節名を再確認し、§2-2で、**実際に学習済みpolicyを1つ読み込み、
「最終層の重み・バイアスに、どうやってアクセスするか」を、推測せず、直接確認**します。


In [ ]:
import math
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import load_env_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100
RIGHT_FACING_YAW_RAD = -math.pi / 2

_probe_cfg = load_env_cfg(TASK, play=True)
_probe_cfg.scene.num_envs = 1
_probe_env = ManagerBasedRlEnv(cfg=_probe_cfg, device=DEVICE, render_mode=None)
_robot = _probe_env.scene["robot"]

JOINT_NAMES = list(_robot.joint_names)
assert len(JOINT_NAMES) == 29
print("関節名(29個)を確認しました。")
for i, name in enumerate(JOINT_NAMES):
    print(f"  [{i:2d}] {name}")

_probe_env.close()
del _probe_env
torch.cuda.empty_cache()


Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be changed or removed without notice.
Module mujoco_warp._src.smooth f756680 load on device 'cuda:0' took 7046.06 ms  (compiled)
Module mujoco_warp._src.collision_driver 4d8b905 load on device 'cuda:0' took 141.91 ms  (compiled)
Module _nxn_broadphase__locals__kernel_e916423b e916423 load on device 'cuda:0' took 529.78 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_f33437d9 f33437d load on device 'cuda:0' took 1094.03 ms  (compiled)
Module mujoco_warp._src.constraint 63c97cb load on device 'cuda:0' took 3677.20 ms  (compiled)
Module mujoco_warp._src.forward 7114b3d load on device 'cuda:0'

In [ ]:
# --- actorの内部構造へのアクセス方法を、実際に確認する ---
import glob, re
from pathlib import Path
from dataclasses import asdict
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_rl_cfg

# 適当な1つのcheckpoint(seed1)を使って、runnerの構造を調べる
_run_dirs = sorted(glob.glob("logs/rsl_rl/g1_velocity/*_seed1"))
assert _run_dirs, "seed1のrunディレクトリが見つかりません。学習済みcheckpointが必要です。"
_ckpt = max(Path(_run_dirs[-1]).glob("model_*.pt"),
            key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

_play_cfg = load_env_cfg(TASK, play=True)
_play_cfg.scene.num_envs = 1
_play_cfg.seed = ENV_SEED
_agent_cfg = load_rl_cfg(TASK)

_raw_env = ManagerBasedRlEnv(cfg=_play_cfg, device=DEVICE, render_mode=None)
_env = RslRlVecEnvWrapper(_raw_env, clip_actions=_agent_cfg.clip_actions)
_runner = MjlabOnPolicyRunner(_env, asdict(_agent_cfg), device=DEVICE)
_runner.load(str(_ckpt), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
_policy = _runner.get_inference_policy(device=DEVICE)

# runner・policyの構造を、実際に表示して確認する
print("runner.alg の型:", type(_runner.alg))
print("runner.alg.actor の型:", type(_runner.alg.actor))
print("runner.alg.actor is get_inference_policy()の戻り値 と同じオブジェクトか:",
      _runner.alg.actor is _runner.get_inference_policy(device=DEVICE))
print()
print("runner.alg.actor.mlp の構造:")
print(_runner.alg.actor.mlp)
print()
# mlpの最終層(出力層)への、実際のアクセスパスを確認する
_actor_mlp = _runner.alg.actor.mlp
print("actor.mlp の中身(Sequentialの各層):")
for idx, layer in enumerate(_actor_mlp):
    print(f"  [{idx}] {layer}")


Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [ ]:
# 最終層(出力層、29次元)の重み・バイアスの形を確認する
# Sequential構造は、通常 [Linear, ELU, Linear, ELU, Linear, ELU, Linear] の7要素のはず。
# 最後の要素(index -1)が、出力層(Linear(128, 29))であることを確認する。
_final_layer = _actor_mlp[-1]
print("最終層:", _final_layer)
print("weight の形状:", _final_layer.weight.shape)  # 期待値: (29, 128)
print("bias の形状:", _final_layer.bias.shape)      # 期待値: (29,)

assert _final_layer.weight.shape[0] == 29, "最終層の出力次元が29ではありません。構造を再確認してください。"
print("\n✅ 最終層への、正しいアクセスパスを確認しました: runner.alg.actor.mlp[-1]")

_env.close()
del _runner, _policy, _env, _raw_env
torch.cuda.empty_cache()


最終層: Linear(in_features=128, out_features=29, bias=True)
weight の形状: torch.Size([29, 128])
bias の形状: torch.Size([29])

✅ 最終層への、正しいアクセスパスを確認しました: runner.alg.actor.mlp[-1]


## 3. ablation関数の定義

**「関節iの出力を除去する」=「最終層の i 行目の重みとバイアスを0で上書きする」**。出力層には活性化関数がなく、行動のクリップもないため、除去した関節の目標角度はデフォルト角度にちょうど一致します。policyの残りの部分は、通常どおり動き続けます(1か所だけの遮断)。


In [ ]:
import numpy as np
import pandas as pd


def load_policy_with_ablation(checkpoint_path, ablate_joint_idx=None, env_seed=ENV_SEED,
                                yaw_rad=RIGHT_FACING_YAW_RAD):
    """policyを読み込み、必要ならablate_joint_idx番目の出力ユニットを無効化する"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (yaw_rad, yaw_rad)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    if ablate_joint_idx is not None:
        with torch.no_grad():
            # 検証済みの正しい経路: runner.alg.actor は、get_inference_policy() の
            # 戻り値と同一オブジェクト(is で確認済み)。runner.alg.policy という
            # 属性は存在しないので注意。
            final_layer = runner.alg.actor.mlp[-1]
            final_layer.weight[ablate_joint_idx, :] = 0.0
            final_layer.bias[ablate_joint_idx] = 0.0

    return runner, policy, env, raw_env


def run_ablation_rollout(checkpoint_path, ablate_joint_idx, forward_speed_mps, steps=250,
                           env_seed=ENV_SEED):
    """1回のablationロールアウトを実行し、転倒の有無と歩容の要約を返す"""
    runner, policy, env, raw_env = load_policy_with_ablation(
        checkpoint_path, ablate_joint_idx=ablate_joint_idx, env_seed=env_seed
    )
    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)

    obs = env.get_observations()
    fell_over = False
    base_x_start = None
    base_x_end = None
    roll_vals = []

    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)

        qpos = raw_env.sim.data.qpos[0].cpu().numpy()
        if base_x_start is None:
            base_x_start = qpos[0]
        base_x_end = qpos[0]

        w, x, y, z = qpos[3:7]
        roll = np.degrees(np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)))
        roll_vals.append(roll)

        if bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0]):
            fell_over = True
            break

    n_steps_completed = step + 1
    distance = base_x_end - base_x_start
    roll_range = max(roll_vals) - min(roll_vals) if roll_vals else float("nan")

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {
        "fell_over": fell_over,
        "n_steps_completed": n_steps_completed,
        "distance_m": round(float(distance), 4),
        "roll_range_deg": round(float(roll_range), 2),
    }


## 4. 第1段階: スクリーニング(29関節 x 少数seed、1速度)

**まず、速度0.5m/s固定、seed 1・2・3の、3つだけを対象に、29関節すべてを、1つずつablation**します
(29関節 x 3seed = 87本)。ここで「転倒が起きた関節」があれば、**次の段階で、その関節に絞って、
8seed全部・速度スイープの、詳しい検証**に進みます。

**所要時間の目安**: 87本のロールアウト(1本あたり数十秒程度)なので、全体で1〜2時間程度を
見込んでください。


In [ ]:
SCREEN_SEEDS = [1, 2, 3]
SCREEN_SPEED_MPS = 0.5

screening_rows = []

for seed in SCREEN_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    # まず、ablationなし(baseline)を1回確認しておく
    print(f"=== seed{seed}: baseline(ablationなし) ===")
    result = run_ablation_rollout(ckpt, ablate_joint_idx=None, forward_speed_mps=SCREEN_SPEED_MPS)
    screening_rows.append({"seed": seed, "joint_idx": -1, "joint_name": "(baseline)", **result})
    print(f"  {result}")

    for joint_idx, joint_name in enumerate(JOINT_NAMES):
        print(f"=== seed{seed}: ablate [{joint_idx}] {joint_name} ===")
        result = run_ablation_rollout(ckpt, ablate_joint_idx=joint_idx, forward_speed_mps=SCREEN_SPEED_MPS)
        screening_rows.append({"seed": seed, "joint_idx": joint_idx, "joint_name": joint_name, **result})
        print(f"  {result}")

screening_df = pd.DataFrame(screening_rows)
screening_df.to_csv("g1_ablation_screening.csv", index=False)
print("\n完了。g1_ablation_screening.csv に保存しました。")


## 5. 結果の集計・criticalityランキング

In [ ]:
# 各関節ごとに、3seed中、何回転倒したかを集計する
ablation_only = screening_df[screening_df["joint_idx"] >= 0].copy()

criticality = (
    ablation_only.groupby(["joint_idx", "joint_name"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_trials"})
    .reset_index()
    .sort_values("n_falls", ascending=False)
)
criticality["fall_rate"] = criticality["n_falls"] / criticality["n_trials"]

print("=== Criticalityランキング(転倒回数が多い順) ===")
print(criticality.to_string(index=False))

criticality.to_csv("g1_ablation_criticality_ranking.csv", index=False)

n_critical = (criticality["n_falls"] > 0).sum()
print(f"\n29関節中、{n_critical}個の関節で、少なくとも1回、転倒が確認されました。")
if n_critical > 0:
    print("次の段階(深掘り)では、これらの関節を対象に、8seed全部・速度スイープで検証することを推奨します:")
    print(criticality[criticality["n_falls"] > 0]["joint_name"].tolist())
else:
    print("この速度・このseed数では、転倒を引き起こす単一関節ablationは、見つかりませんでした。")
    print("速度をさらに上げる、またはseed数を増やすことを検討してください。")


## 6. 第2段階: 深掘り(左右対称性の検証、4関節 x 速度スイープ x 8seed)

スクリーニング(§4・§5)で見つかった、上位2関節(`right_ankle_pitch_joint`、
`right_hip_pitch_joint`)に加え、**左右対称性を正しく検証するため、対応する左側
(`left_ankle_pitch_joint`、`left_hip_pitch_joint`)も、必ず含めて**、4関節で
比較します。

- **速度**: 0.2〜1.0 m/s(0.1刻み、9段階)
- **seed**: 8個全部
- **関節**: 4個(左右2ペア)
- **合計**: 4 x 9 x 8 = 288 本

**所要時間の目安**: スクリーニング(90本)の、約3倍の規模なので、**数時間程度**を
見込んでください。

> **クラッシュ対策(resume可能な設計)**: 以下のセルは、**1本のロールアウトが終わるたびに、
> 即座にCSVへ追記**します。また、**20本ごとに、Google Driveへも自動的に退避**します。
> **もしセッションがクラッシュ・再接続しても、同じセルを、そのままもう一度実行すれば、
> 既に完了した組み合わせは自動的にスキップされ、続きからだけ実行されます**
> (学習ノートブックの、resumeを廃止した設計とは異なり、こちらは1本ずつの独立した
> ロールアウトの繰り返しなので、"チャンク分割"ではなく、"完了済みをスキップする"という、
> より単純な方式で、確実にresumeできます)。
>
> **事前準備**: このセルを実行する前に、Google Driveがマウントされている必要があります。
> まだの場合は、先に以下を実行してください。
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> ```


In [ ]:
DEEPDIVE_JOINTS = [
    "right_ankle_pitch_joint",
    "left_ankle_pitch_joint",
    "right_hip_pitch_joint",
    "left_hip_pitch_joint",
]
DEEPDIVE_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
DEEPDIVE_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

DEEPDIVE_CSV = "g1_ablation_deepdive.csv"
DRIVE_BACKUP_EVERY_N = 20  # この本数ごとに、Google Driveへ退避する
DRIVE_BACKUP_PATH = "/content/drive/MyDrive/g1_ablation_deepdive_backup"

import os
os.makedirs(DRIVE_BACKUP_PATH, exist_ok=True)


def backup_csv_to_drive():
    import shutil
    shutil.copy(DEEPDIVE_CSV, f"{DRIVE_BACKUP_PATH}/g1_ablation_deepdive.csv")
    print(f"  [Drive退避完了] {DRIVE_BACKUP_PATH}/g1_ablation_deepdive.csv")


# --- 再開(resume)のための準備 ---
# 既に g1_ablation_deepdive.csv が存在すれば、それを読み込み、
# 「既に完了している(joint_name, seed, speed_mps)の組み合わせ」を把握する。
# これにより、セッションがクラッシュ・再接続した後に、このセルを再実行しても、
# 完了済みの分は飛ばして、続きからだけ実行される。
if os.path.exists(DEEPDIVE_CSV):
    _existing_df = pd.read_csv(DEEPDIVE_CSV)
    completed_combos = set(
        zip(_existing_df["joint_name"], _existing_df["seed"], _existing_df["speed_mps"])
    )
    print(f"既存のログを検出しました: {len(_existing_df)} 行、"
          f"{len(completed_combos)} 個の組み合わせが完了済みです。続きから再開します。")
else:
    completed_combos = set()
    print("新規に開始します。")

total_combos = len(DEEPDIVE_JOINTS) * len(DEEPDIVE_SEEDS) * len(DEEPDIVE_SPEEDS)
n_done_this_session = 0

for joint_name in DEEPDIVE_JOINTS:
    joint_idx = JOINT_NAMES.index(joint_name)

    for seed in DEEPDIVE_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in DEEPDIVE_SPEEDS:
            combo_key = (joint_name, seed, speed)
            if combo_key in completed_combos:
                continue  # 既に完了済みなので、スキップする

            n_done_total = len(completed_combos) + n_done_this_session + 1
            print(f"=== [{n_done_total}/{total_combos}] {joint_name}(idx={joint_idx}) / "
                  f"seed{seed} / {speed:.1f}m/s ===")
            result = run_ablation_rollout(ckpt, ablate_joint_idx=joint_idx,
                                            forward_speed_mps=speed)
            print(f"  {result}")

            # --- 1本終わるごとに、即座にCSVへ追記する(クラッシュ対策の核心) ---
            row_df = pd.DataFrame([{
                "joint_name": joint_name, "joint_idx": joint_idx,
                "seed": seed, "speed_mps": speed, **result
            }])
            write_header = not os.path.exists(DEEPDIVE_CSV)
            row_df.to_csv(DEEPDIVE_CSV, mode="a", header=write_header, index=False)

            n_done_this_session += 1
            if n_done_this_session % DRIVE_BACKUP_EVERY_N == 0:
                backup_csv_to_drive()

# 最後に、必ず一度、Driveへ退避しておく
backup_csv_to_drive()

deepdive_df = pd.read_csv(DEEPDIVE_CSV)
print(f"\n完了。{DEEPDIVE_CSV} に、合計 {len(deepdive_df)} 行、保存しました。")


## 7. 結果の分析:左右対称性・速度依存性の確認

In [ ]:
# 関節 x 速度 ごとの、8seed中の転倒率を集計する
pivot = (
    deepdive_df.groupby(["joint_name", "speed_mps"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
pivot["fall_rate"] = pivot["n_falls"] / pivot["n_seeds"]

print("=== 関節 x 速度 ごとの転倒率 ===")
table = pivot.pivot(index="speed_mps", columns="joint_name", values="fall_rate")
print(table.to_string())

table.to_csv("g1_ablation_deepdive_summary_table.csv")

print("\n=== 各関節の、最初に転倒が現れた速度(onset) ===")
for joint_name in DEEPDIVE_JOINTS:
    sub = pivot[(pivot["joint_name"] == joint_name) & (pivot["fall_rate"] > 0)]
    if len(sub) > 0:
        onset = sub["speed_mps"].min()
        print(f"  {joint_name}: onset = {onset:.1f} m/s")
    else:
        print(f"  {joint_name}: この速度範囲(0.2-1.0m/s)では、転倒は確認されませんでした")

print("\n=== 左右差の比較 ===")
for pair_base in ["ankle_pitch", "hip_pitch"]:
    right_name = f"right_{pair_base}_joint"
    left_name = f"left_{pair_base}_joint"
    right_sub = pivot[(pivot["joint_name"] == right_name) & (pivot["fall_rate"] > 0)]
    left_sub = pivot[(pivot["joint_name"] == left_name) & (pivot["fall_rate"] > 0)]
    right_onset = right_sub["speed_mps"].min() if len(right_sub) > 0 else float("inf")
    left_onset = left_sub["speed_mps"].min() if len(left_sub) > 0 else float("inf")
    print(f"  {pair_base}: 右 onset={right_onset}, 左 onset={left_onset}"
          f"  -> {'右がより脆弱' if right_onset < left_onset else ('左がより脆弱' if left_onset < right_onset else '左右差なし')}")


## 8. 左右対称性の、頑健な比較(教訓を反映した、正式な分析)

§7では、「8seed中で、最も転倒しやすかった速度(onsetの最小値)」を、そのまま比較に
使っていました。しかし、これは**1つの外れ値(特定のseedだけの、たまたまの結果)に、
比較全体が引きずられてしまう**という、重大な弱点を持つことが分かりました
(実際に、hip_pitchで、seed6という単一の例外により、「左がより脆弱」という、
誤った結論に至りかけました)。

このセクションでは、**2つの、より頑健な指標**で、改めて左右差を比較します。

1. **多数決による比較**: 「onsetが低い方」を、8seedそれぞれで判定し、**その多数決**を見る
2. **生存時間による比較**: 「転倒するかどうか」ではなく、**「転倒するまでに、何ステップ
   持ちこたえたか」**を、左右で比較する。特に、`ankle_pitch`のように、onsetでは
   ほぼ全seedが同着になってしまう(=onsetでは比較できない)関節に、有効な視点


In [ ]:
def compare_left_right_by_majority(df, pair_base, seeds):
    # 「onsetが低い方」を、seedごとに判定し、その多数決を集計する(外れ値に頑健)
    right_name = f"right_{pair_base}_joint"
    left_name = f"left_{pair_base}_joint"
    right_more_fragile = 0
    left_more_fragile = 0
    tied_or_undetermined = 0

    for seed in seeds:
        r_falls = df[(df["joint_name"] == right_name) & (df["seed"] == seed) & (df["fell_over"])]
        l_falls = df[(df["joint_name"] == left_name) & (df["seed"] == seed) & (df["fell_over"])]
        r_onset = r_falls["speed_mps"].min() if len(r_falls) > 0 else float("inf")
        l_onset = l_falls["speed_mps"].min() if len(l_falls) > 0 else float("inf")

        if r_onset < l_onset:
            right_more_fragile += 1
        elif l_onset < r_onset:
            left_more_fragile += 1
        else:
            tied_or_undetermined += 1

    n = len(seeds)
    print(f"{pair_base}: 右がより脆弱={right_more_fragile}/{n}, "
          f"左がより脆弱={left_more_fragile}/{n}, 同等/判定不能={tied_or_undetermined}/{n}")
    return right_more_fragile, left_more_fragile, tied_or_undetermined


def compare_left_right_by_survival(df, pair_base, seeds, speed):
    # 指定した速度における、転倒までの生存ステップ数を、左右で比較する
    right_name = f"right_{pair_base}_joint"
    left_name = f"left_{pair_base}_joint"

    right_steps = df[(df["joint_name"] == right_name) & (df["speed_mps"] == speed)]["n_steps_completed"]
    left_steps = df[(df["joint_name"] == left_name) & (df["speed_mps"] == speed)]["n_steps_completed"]

    print(f"{pair_base} (速度{speed}m/s時点、生存ステップ数):")
    print(f"  右: 平均={right_steps.mean():.1f}  範囲={right_steps.min()}-{right_steps.max()}")
    print(f"  左: 平均={left_steps.mean():.1f}  範囲={left_steps.min()}-{left_steps.max()}")
    return right_steps.mean(), left_steps.mean()


print("=== 多数決による、左右差の判定 ===")
compare_left_right_by_majority(deepdive_df, "hip_pitch", DEEPDIVE_SEEDS)
compare_left_right_by_majority(deepdive_df, "ankle_pitch", DEEPDIVE_SEEDS)

print()
print("=== 生存時間による、左右差の判定(onsetでは判定できなかった関節を、補完する) ===")
compare_left_right_by_survival(deepdive_df, "ankle_pitch", DEEPDIVE_SEEDS, speed=0.2)
compare_left_right_by_survival(deepdive_df, "hip_pitch", DEEPDIVE_SEEDS, speed=0.5)


## 9. 結果のまとめ

この節の旧版のまとめは、撤回した機序の知見(デフォルト姿勢への受動的な崩壊)を含んでいたため削除しました。確定した結果は改訂稿の§3.1–3.4にあります。

- 3シードのスクリーニングで転倒を生じたのは3関節(右足首ピッチ、左足首ピッチ、右股関節ピッチ)。8シードに広げると、左股関節ピッチと右膝が加わり5関節(§13)。
- 足首ピッチ:0.2 m/sでの生存時間は右の方が短い(8方策中7方策で左が長い)。
- 股関節ピッチ:転倒開始速度は右の方が低い傾向(統計的には未確立)。
- 段階的減衰:足首ピッチは出力の減少とともに転倒率がなめらかに上がり、右の方が早く始まる。股関節ピッチは完全除去でのみ転倒(§16)。


## 10. 機序の解明:動画観察 + 関節角度の時系列解析

> **Superseded / 撤回済み (see the revised manuscript, §2.4 and §3.3).** This section read joint angles after the simulator's automatic reset, so the "convergence to the default posture" it reports is a measurement artifact. The corrected analysis is in `02_remeasure_mechanism_right_knee_1p0_screen.ipynb`. The code is kept unchanged for transparency.
> この節は自動リセット後の関節角度を読んでいたため、「デフォルト姿勢への収束」は計測上のアーティファクトです。訂正後の解析はnotebook 02にあります。透明性のためコードは変更せずに残しています。


**まず動画で定性的に観察し、次に、時系列データを
用いて、機序を定量的に確認**します。

対象は、**最も深刻だった`right_ankle_pitch_joint`のablation**です(8seed全てが、
0.2m/sという、最も遅い速度から、平均97ステップ程度で転倒)。比較のため、
`right_hip_pitch_joint`(用量反応的なパターンを示した関節)も、併せて確認します。

**まず、"どちら向きに崩れるか"という、基本的な問いを立てます。** ankle_pitch
(底背屈)は、歩行における、前後方向のバランス・推進に関わる関節なので、
**前後方向(pitch方向)への
転倒が起きているのではないか**、という仮説を、まず持って臨みます。


In [ ]:
import mediapy as media

Path("videos_mechanism").mkdir(exist_ok=True)


def make_ablation_video(joint_name, seed, speed_mps, out_path, steps=150):
    joint_idx = JOINT_NAMES.index(joint_name)
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = ENV_SEED
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode="rgb_array")
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(ckpt), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    with torch.no_grad():
        final_layer = runner.alg.actor.mlp[-1]
        final_layer.weight[joint_idx, :] = 0.0
        final_layer.bias[joint_idx] = 0.0

    forward_cmd = torch.tensor([[speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    frames = []
    for _ in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)
        frame = np.asarray(raw_env.render())
        frames.append(frame[0] if frame.ndim == 4 else frame)
        if bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0]):
            break

    media.write_video(out_path, frames, fps=int(raw_env.metadata.get("render_fps", 50)))
    print(f"  保存しました: {out_path} ({len(frames)} フレーム)")

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()


# right_ankle_pitch(最も深刻)を、代表的な2seedで動画化
for seed in [1, 2]:
    print(f"=== right_ankle_pitch_joint / seed{seed} / 0.2m/s ===")
    make_ablation_video("right_ankle_pitch_joint", seed, 0.2,
                         f"videos_mechanism/ankle_pitch_seed{seed}.mp4")

# 比較として、right_hip_pitch(onset付近の0.6m/s)も、1seedだけ動画化
print("=== right_hip_pitch_joint / seed1 / 0.6m/s ===")
make_ablation_video("right_hip_pitch_joint", 1, 0.6,
                     "videos_mechanism/hip_pitch_seed1.mp4")


In [ ]:
from IPython.display import Video, display

for path in ["videos_mechanism/ankle_pitch_seed1.mp4",
             "videos_mechanism/ankle_pitch_seed2.mp4",
             "videos_mechanism/hip_pitch_seed1.mp4"]:
    if os.path.exists(path):
        print(f"--- {path} ---")
        display(Video(path, embed=True, width=480))


## 11. 関節角度・base姿勢の、時系列解析(定量確認)

> **Superseded / 撤回済み (see the revised manuscript, §2.4 and §3.3).** This section read joint angles after the simulator's automatic reset, so the "convergence to the default posture" it reports is a measurement artifact. The corrected analysis is in `02_remeasure_mechanism_right_knee_1p0_screen.ipynb`. The code is kept unchanged for transparency.
> この節は自動リセット後の関節角度を読んでいたため、「デフォルト姿勢への収束」は計測上のアーティファクトです。訂正後の解析はnotebook 02にあります。透明性のためコードは変更せずに残しています。


動画での観察を踏まえ、**base(胴体)のpitch角度・roll角度・高さの時系列**を、
実際に記録し、**「前後方向(pitch)に崩れているのか、左右方向(roll)に崩れているのか」**
を、定量的に確認します。

**「崩れる直前の数ステップで、
どの角度が、どれだけ急激に変化しているか」**を見ます。


In [ ]:
def run_ablation_rollout_with_full_log(checkpoint_path, ablate_joint_idx, forward_speed_mps,
                                          steps=250, env_seed=ENV_SEED, extra_joint_names=None):
    # run_ablation_rolloutと同じだが、summaryだけでなく、全ステップの時系列を返す
    # 拡張版: yaw(回旋)・y(左右方向の位置)・指定した関節の実測角度も、併せて記録する
    runner, policy, env, raw_env = load_policy_with_ablation(
        checkpoint_path, ablate_joint_idx=ablate_joint_idx, env_seed=env_seed
    )
    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    extra_joint_names = extra_joint_names or []
    extra_joint_idxs = [JOINT_NAMES.index(n) for n in extra_joint_names]

    obs = env.get_observations()
    rows = []
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)

        qpos = raw_env.sim.data.qpos[0].cpu().numpy()
        w, x, y, z = qpos[3:7]
        roll = np.degrees(np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)))
        pitch = np.degrees(np.arcsin(np.clip(2*(w*y - z*x), -1.0, 1.0)))
        yaw = np.degrees(np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z)))

        row = {
            "step": step, "base_x": qpos[0], "base_y": qpos[1], "base_z": qpos[2],
            "base_roll_deg": roll, "base_pitch_deg": pitch, "base_yaw_deg": yaw,
        }

        if extra_joint_idxs:
            joint_pos = raw_env.scene["robot"].data.joint_pos[0].detach().cpu().numpy()
            for name, idx in zip(extra_joint_names, extra_joint_idxs):
                row[f"jpos_{name}_deg"] = np.degrees(joint_pos[idx])

        rows.append(row)

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return pd.DataFrame(rows)


# --- right_ankle_pitch: 8seed全部、右膝・右足首自体の実測角度も併せて記録する ---
# (動画観察: seed1=つま先の引っかかり→前方転倒、seed2=過剰背屈→膝折れ→転倒、
#  という、2種類の異なる破綻パターンが見られたため、これを定量的に確認する)
ankle_pitch_idx = JOINT_NAMES.index("right_ankle_pitch_joint")
mechanism_logs_ankle = {}
for seed in [1, 2, 3, 4, 5, 6, 7, 8]:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    print(f"=== right_ankle_pitch / seed{seed} / 0.2m/s の時系列を記録中 ===")
    df = run_ablation_rollout_with_full_log(
        ckpt, ankle_pitch_idx, forward_speed_mps=0.2,
        extra_joint_names=["right_ankle_pitch_joint", "right_knee_joint"]
    )
    mechanism_logs_ankle[seed] = df
    print(f"  {len(df)} ステップ分、記録しました(転倒: {len(df) < 250}）")


# --- right_hip_pitch: 複数seedで、回旋(yaw)・左右方向の偏り(y)・左右hip_pitchの実測角度を記録する ---
# (動画観察: 小刻みなステップで右方向に回旋、体重が右へ偏る、という所見を確認する)
hip_pitch_idx = JOINT_NAMES.index("right_hip_pitch_joint")
mechanism_logs_hip = {}
for seed in [1, 3, 6, 8]:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    print(f"=== right_hip_pitch / seed{seed} / 0.5m/s の時系列を記録中 ===")
    df = run_ablation_rollout_with_full_log(
        ckpt, hip_pitch_idx, forward_speed_mps=0.5,
        extra_joint_names=["right_hip_pitch_joint", "left_hip_pitch_joint"]
    )
    mechanism_logs_hip[seed] = df
    print(f"  {len(df)} ステップ分、記録しました(転倒: {len(df) < 250}）")


In [ ]:
# === right_ankle_pitch: 「つま先の引っかかり(遊脚相)」か「膝折れ(立脚相)」か、seedごとに分類する ===
print("=== right_ankle_pitch: 転倒直前(最後の10ステップ)の、右膝・右足首の実測角度 ===")
ankle_mechanism_summary = []
for seed, df in mechanism_logs_ankle.items():
    tail = df.tail(10)
    knee_col = "jpos_right_knee_joint_deg"
    ankle_col = "jpos_right_ankle_pitch_joint_deg"
    if knee_col not in tail.columns:
        continue
    knee_start, knee_end = tail[knee_col].iloc[0], tail[knee_col].iloc[-1]
    ankle_start, ankle_end = tail[ankle_col].iloc[0], tail[ankle_col].iloc[-1]
    knee_change = knee_end - knee_start
    ankle_change = ankle_end - ankle_start

    # 膝折れ(knee buckling): 膝が、転倒直前に、急激に"曲がる"方向へ変化する
    # 過剰背屈(over-dorsiflexion): 足首が、大きく背屈方向(符号は要確認)へ変化する
    print(f"  seed{seed}: 膝角度変化={knee_change:+.1f}度(開始{knee_start:.1f}->終了{knee_end:.1f}), "
          f"足首角度変化={ankle_change:+.1f}度(開始{ankle_start:.1f}->終了{ankle_end:.1f}), "
          f"n_steps={len(df)}")
    ankle_mechanism_summary.append({
        "seed": seed, "n_steps": len(df),
        "knee_change_deg": knee_change, "ankle_change_deg": ankle_change,
    })

ankle_mechanism_df = pd.DataFrame(ankle_mechanism_summary)
print()
print("参考: n_stepsが短い(早く転倒した)seedほど、動画のseed1型(引っかかり)に近いか、")
print("      seed2型(膝折れ)に近いかは、上記の膝・足首の変化パターンから、判断してください。")


# === right_hip_pitch: 回旋(yaw)・左右方向の偏り(base_y)を確認する ===
print()
print("=== right_hip_pitch: 転倒(または完走)直前の、base_yaw・base_yの推移 ===")
for seed, df in mechanism_logs_hip.items():
    tail = df.tail(20)
    yaw_change = tail["base_yaw_deg"].iloc[-1] - tail["base_yaw_deg"].iloc[0]
    y_change = tail["base_y"].iloc[-1] - tail["base_y"].iloc[0]
    fell = len(df) < 250
    print(f"  seed{seed}: yaw変化={yaw_change:+.1f}度, y方向の変位={y_change:+.3f}m, "
          f"n_steps={len(df)}, 転倒={fell}")
    if yaw_change != 0:
        direction = "右回旋(時計回り)" if yaw_change < 0 else "左回旋(反時計回り)"
        # 注: yaw角度の符号とロボットの回転方向の対応は、座標系依存のため、
        # 実際の動画観察(「右方向に回旋」)と、機械的に一致するか、必ず目視でも確認すること
        print(f"    (yawの符号から見た回転方向: {direction} ※座標系依存、動画と併せて確認)")


## 12. 追加実験1: hip_pitchの機序確認(デフォルト姿勢への崩壊か)

> **Superseded / 撤回済み (see the revised manuscript, §2.4 and §3.3).** This section read joint angles after the simulator's automatic reset, so the "convergence to the default posture" it reports is a measurement artifact. The corrected analysis is in `02_remeasure_mechanism_right_knee_1p0_screen.ipynb`. The code is kept unchanged for transparency.
> この節は自動リセット後の関節角度を読んでいたため、「デフォルト姿勢への収束」は計測上のアーティファクトです。訂正後の解析はnotebook 02にあります。透明性のためコードは変更せずに残しています。


`right_ankle_pitch`で確立した方法(転倒直前の関節角度と、`default_joint_pos`との比較)を、
`right_hip_pitch`にも適用する。§6の深掘り実験では、0.5m/sで転倒したのは、4seed中
seed1のみだったため、**より高い速度(転倒率の高い0.8m/s)で、8seed全部から、
追加の転倒事例を収集**した上で、判定する。


In [ ]:
# 0.8m/s(deepdive_dfで、転倒率が高かった速度)で、8seed全部、hip_pitchの時系列を記録する
hip_pitch_idx = JOINT_NAMES.index("right_hip_pitch_joint")
mechanism_logs_hip_08 = {}
for seed in [1, 2, 3, 4, 5, 6, 7, 8]:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    print(f"=== right_hip_pitch / seed{seed} / 0.8m/s の時系列を記録中 ===")
    df = run_ablation_rollout_with_full_log(
        ckpt, hip_pitch_idx, forward_speed_mps=0.8,
        extra_joint_names=["right_hip_pitch_joint"]
    )
    mechanism_logs_hip_08[seed] = df
    print(f"  {len(df)} ステップ分、記録しました(転倒: {len(df) < 250}）")


In [ ]:
# 環境が実際に使っている、right_hip_pitchのデフォルト角度を確認する
_probe_cfg6 = load_env_cfg(TASK, play=True)
_probe_cfg6.scene.num_envs = 1
_probe_env6 = ManagerBasedRlEnv(cfg=_probe_cfg6, device=DEVICE, render_mode=None)
default_joint_pos = _probe_env6.scene["robot"].data.default_joint_pos[0].cpu().numpy()
hip_pitch_default_deg = np.degrees(default_joint_pos[JOINT_NAMES.index("right_hip_pitch_joint")])
print(f"right_hip_pitch_joint の、デフォルト角度: {hip_pitch_default_deg:.2f} 度")
_probe_env6.close()
del _probe_env6
torch.cuda.empty_cache()

# 転倒した各seedについて、転倒直前(最後の10ステップ)の、右hip_pitch実測角度の平均を、
# デフォルト角度と比較する
print()
print("=== 転倒した各seedの、直前(最後の10ステップ)の、右hip_pitch実測角度 ===")
fell_count = 0
for seed, df in mechanism_logs_hip_08.items():
    fell = len(df) < 250
    if not fell:
        print(f"  seed{seed}: 転倒せず(250ステップ完走)。対象外。")
        continue
    fell_count += 1
    tail = df.tail(10)
    col = "jpos_right_hip_pitch_joint_deg"
    terminal_val = tail[col].iloc[-1]
    diff_from_default = terminal_val - hip_pitch_default_deg
    print(f"  seed{seed}: 転倒直前の実測角度 = {terminal_val:.2f}度  "
          f"(デフォルト角度との差: {diff_from_default:+.2f}度)、n_steps={len(df)}")

print()
if fell_count == 0:
    print("この速度(0.8m/s)でも、転倒したseedがありませんでした。速度をさらに上げて、再試行してください。")
else:
    print(f"{fell_count}個のseedで、転倒直前の実測角度を、デフォルト角度と比較しました。")
    print("差が小さい(概ね数度以内)seedが多ければ、ankle_pitchと同じ「デフォルト姿勢への崩壊」")
    print("機序が支持されます。差が大きければ、hip_pitchは、別の機序(例えば動画で見られた")
    print("「回旋を伴う崩れ」)により、異なる終着点に至っている可能性が高くなります。")


## 13. 追加実験2: スクリーニングの8seed拡張

§4のスクリーニング(29関節、3seedのみ)を、8seedに拡張する。ただし、**§6の深掘り実験で、
既に8seed分のデータがある4関節(左右のankle_pitch・hip_pitch)は、再実行せず、
`deepdive_df`(速度0.5m/sの行)から、該当データを再利用**する。これにより、
**新たに実行が必要なのは、残り25関節 × 追加5seed(4〜8)= 125本のみ**で済む。

**resume可能な設計(§6と同じ考え方)**: 1本終わるごとに、即座にCSVへ追記し、
既存のCSVがあれば、完了済みの組み合わせを自動的にスキップする。


In [ ]:
SCREEN_EXT_CSV = "g1_ablation_screening_extended.csv"
SCREEN_EXT_SPEED = 0.5
SCREEN_EXT_NEW_SEEDS = [4, 5, 6, 7, 8]  # 追加する5seed
ALREADY_COVERED_JOINTS = set(DEEPDIVE_JOINTS)  # §6で、既に8seed分あるので、再実行不要
JOINTS_TO_SCREEN_EXT = [j for j in JOINT_NAMES if j not in ALREADY_COVERED_JOINTS]
print(f"新たにスクリーニングする関節数: {len(JOINTS_TO_SCREEN_EXT)}"
      f"(既に8seed分あるため除外: {sorted(ALREADY_COVERED_JOINTS)})")

# --- resumeのための準備(既存CSVがあれば、完了済みの組み合わせを把握する) ---
if os.path.exists(SCREEN_EXT_CSV):
    _existing_ext_df = pd.read_csv(SCREEN_EXT_CSV)
    ext_completed = set(zip(_existing_ext_df["joint_name"], _existing_ext_df["seed"]))
    print(f"既存のログを検出: {len(_existing_ext_df)}行、{len(ext_completed)}組が完了済み")
else:
    ext_completed = set()
    print("新規に開始します。")

total_ext_combos = len(JOINTS_TO_SCREEN_EXT) * len(SCREEN_EXT_NEW_SEEDS)
n_done = 0

for joint_name in JOINTS_TO_SCREEN_EXT:
    joint_idx = JOINT_NAMES.index(joint_name)
    for seed in SCREEN_EXT_NEW_SEEDS:
        if (joint_name, seed) in ext_completed:
            continue
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        n_done += 1
        print(f"=== [{len(ext_completed)+n_done}/{total_ext_combos}] "
              f"{joint_name}(idx={joint_idx}) / seed{seed} / {SCREEN_EXT_SPEED}m/s ===")
        result = run_ablation_rollout(ckpt, ablate_joint_idx=joint_idx,
                                        forward_speed_mps=SCREEN_EXT_SPEED)
        print(f"  {result}")

        row_df = pd.DataFrame([{
            "joint_name": joint_name, "joint_idx": joint_idx,
            "seed": seed, "speed_mps": SCREEN_EXT_SPEED, **result
        }])
        write_header = not os.path.exists(SCREEN_EXT_CSV)
        row_df.to_csv(SCREEN_EXT_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{SCREEN_EXT_CSV} に保存しました。")


In [ ]:
# --- 3つのデータソースを統合し、29関節 x 8seedの、完全なcriticalityランキングを作る ---
# 1. 元のスクリーニング(29関節、seed1-3)
# 2. 拡張スクリーニング(25関節、seed4-8。上のセルで作成)
# 3. 深掘り実験(4関節、seed1-8、0.5m/sの行のみ抽出)

screen_ext_df = pd.read_csv(SCREEN_EXT_CSV)

deepdive_at_05 = deepdive_df[deepdive_df["speed_mps"] == SCREEN_EXT_SPEED].copy()
deepdive_at_05 = deepdive_at_05[["joint_name", "joint_idx", "seed", "fell_over"]]

original_screen = screening_df[screening_df["joint_idx"] >= 0][["joint_name", "joint_idx", "seed", "fell_over"]]

full_8seed_screening = pd.concat([
    original_screen,
    screen_ext_df[["joint_name", "joint_idx", "seed", "fell_over"]],
    deepdive_at_05,
], ignore_index=True)

# 重複(同じjoint_name x seedの組み合わせ)があれば、念のため確認する
dup_check = full_8seed_screening.duplicated(subset=["joint_name", "seed"], keep=False)
if dup_check.any():
    print(f"[警告] {dup_check.sum()}行、重複が見つかりました。内容を確認してください:")
    print(full_8seed_screening[dup_check].sort_values(["joint_name", "seed"]))
else:
    print("重複なし。29関節 x 8seed、正しく統合されました。")

full_8seed_screening.to_csv("g1_ablation_screening_full_8seed.csv", index=False)

# criticalityランキングを、8seedベースで、作り直す
full_criticality = (
    full_8seed_screening.groupby(["joint_idx", "joint_name"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_trials"})
    .reset_index()
    .sort_values("n_falls", ascending=False)
)
full_criticality["fall_rate"] = full_criticality["n_falls"] / full_criticality["n_trials"]

print()
print("=== Criticalityランキング(8seedベース、確定版) ===")
print(full_criticality.to_string(index=False))

n_critical_full = (full_criticality["n_falls"] > 0).sum()
print(f"\n29関節中、{n_critical_full}個の関節で、8seed中、少なくとも1回、転倒が確認されました。")
print("(3seedのみのスクリーニングでは3関節だったのに対し、8seedに拡張したことで、"
      "この数字がどう変化したか、確認してください)")


## 14. 追加実験3: left_ankle_pitchの機序確認(左右両方の比較のため)

> **Superseded / 撤回済み (see the revised manuscript, §2.4 and §3.3).** This section read joint angles after the simulator's automatic reset, so the "convergence to the default posture" it reports is a measurement artifact. The corrected analysis is in `02_remeasure_mechanism_right_knee_1p0_screen.ipynb`. The code is kept unchanged for transparency.
> この節は自動リセット後の関節角度を読んでいたため、「デフォルト姿勢への収束」は計測上のアーティファクトです。訂正後の解析はnotebook 02にあります。透明性のためコードは変更せずに残しています。


`right_ankle_pitch`と全く同じ方法を、`left_ankle_pitch`にも適用する。左膝・左足首の
実測角度を、8seed全部で記録し、デフォルト角度との一致を確認する。**§8で確認した通り、
`left_ankle_pitch`は7/8seedで転倒する(0.2m/s)ため、右側とほぼ同じ規模の、
転倒事例が得られるはずである。**


In [ ]:
left_ankle_idx = JOINT_NAMES.index("left_ankle_pitch_joint")
mechanism_logs_left_ankle = {}
for seed in [1, 2, 3, 4, 5, 6, 7, 8]:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    print(f"=== left_ankle_pitch / seed{seed} / 0.2m/s の時系列を記録中 ===")
    df = run_ablation_rollout_with_full_log(
        ckpt, left_ankle_idx, forward_speed_mps=0.2,
        extra_joint_names=["left_ankle_pitch_joint", "left_knee_joint"]
    )
    mechanism_logs_left_ankle[seed] = df
    print(f"  {len(df)} ステップ分、記録しました(転倒: {len(df) < 250}）")


In [ ]:
# 環境が実際に使っている、left_knee・left_ankle_pitchのデフォルト角度を確認する
_probe_cfg7 = load_env_cfg(TASK, play=True)
_probe_cfg7.scene.num_envs = 1
_probe_env7 = ManagerBasedRlEnv(cfg=_probe_cfg7, device=DEVICE, render_mode=None)
default_joint_pos = _probe_env7.scene["robot"].data.default_joint_pos[0].cpu().numpy()
left_knee_default_deg = np.degrees(default_joint_pos[JOINT_NAMES.index("left_knee_joint")])
left_ankle_default_deg = np.degrees(default_joint_pos[JOINT_NAMES.index("left_ankle_pitch_joint")])
print(f"left_knee_joint の、デフォルト角度: {left_knee_default_deg:.2f} 度")
print(f"left_ankle_pitch_joint の、デフォルト角度: {left_ankle_default_deg:.2f} 度")
_probe_env7.close()
del _probe_env7
torch.cuda.empty_cache()

# 転倒した各seedについて、転倒直前(最後の10ステップ)の、開始角度・終了角度を、両方記録する
# (Figure 3の、右側と同じ形式のスロープチャートを作るため、"開始"の値も必要)
print()
print("=== left_ankle_pitch: 転倒した各seedの、左膝の開始・終了角度 ===")
left_ankle_mechanism_rows = []
for seed, df in mechanism_logs_left_ankle.items():
    fell = len(df) < 250
    if not fell:
        print(f"  seed{seed}: 転倒せず(250ステップ完走)。対象外。")
        continue
    tail = df.tail(10)
    knee_col = "jpos_left_knee_joint_deg"
    ankle_col = "jpos_left_ankle_pitch_joint_deg"
    knee_start, knee_end = tail[knee_col].iloc[0], tail[knee_col].iloc[-1]
    ankle_start, ankle_end = tail[ankle_col].iloc[0], tail[ankle_col].iloc[-1]
    print(f"  seed{seed}: 膝 開始={knee_start:.1f}度 -> 終了={knee_end:.1f}度"
          f"(デフォルトとの差: {knee_end - left_knee_default_deg:+.2f}度)、"
          f"足首 開始={ankle_start:.1f}度 -> 終了={ankle_end:.1f}度"
          f"(デフォルトとの差: {ankle_end - left_ankle_default_deg:+.2f}度)、n_steps={len(df)}")
    left_ankle_mechanism_rows.append({
        "seed": seed, "n_steps": len(df),
        "knee_start": knee_start, "knee_end": knee_end,
        "ankle_start": ankle_start, "ankle_end": ankle_end,
    })

left_ankle_mechanism_df = pd.DataFrame(left_ankle_mechanism_rows)
left_ankle_mechanism_df.to_csv("g1_left_ankle_mechanism.csv", index=False)
print()
print("g1_left_ankle_mechanism.csv に保存しました。この内容を、そのまま貼り付けて共有してください。")


## 15. 追加実験4: left_hip_pitchの機序確認(投稿前の、最後の一貫性チェック)

> **Superseded / 撤回済み (see the revised manuscript, §2.4 and §3.3).** This section read joint angles after the simulator's automatic reset, so the "convergence to the default posture" it reports is a measurement artifact. The corrected analysis is in `02_remeasure_mechanism_right_knee_1p0_screen.ipynb`. The code is kept unchanged for transparency.
> この節は自動リセット後の関節角度を読んでいたため、「デフォルト姿勢への収束」は計測上のアーティファクトです。訂正後の解析はnotebook 02にあります。透明性のためコードは変更せずに残しています。


`right_hip_pitch`(§12)と、全く同じ方法を、`left_hip_pitch`にも適用する。
これにより、**ankle_pitch・hip_pitchの、両方の関節ペアで、左右両側の機序確認が揃う**。

`right_hip_pitch`では、0.5m/sでは転倒事例が少なかったため、0.8m/sを使った。
`left_hip_pitch`も、同じ0.8m/sで、8seed全部から、転倒事例を収集する。


In [ ]:
left_hip_idx = JOINT_NAMES.index("left_hip_pitch_joint")
mechanism_logs_left_hip = {}
for seed in [1, 2, 3, 4, 5, 6, 7, 8]:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    print(f"=== left_hip_pitch / seed{seed} / 0.8m/s の時系列を記録中 ===")
    df = run_ablation_rollout_with_full_log(
        ckpt, left_hip_idx, forward_speed_mps=0.8,
        extra_joint_names=["left_hip_pitch_joint"]
    )
    mechanism_logs_left_hip[seed] = df
    print(f"  {len(df)} ステップ分、記録しました(転倒: {len(df) < 250}）")


In [ ]:
# 環境が実際に使っている、left_hip_pitchのデフォルト角度を確認する
# (right_hip_pitchのデフォルトは-17.88度だったが、念のため左側も、実測で確認する)
_probe_cfg8 = load_env_cfg(TASK, play=True)
_probe_cfg8.scene.num_envs = 1
_probe_env8 = ManagerBasedRlEnv(cfg=_probe_cfg8, device=DEVICE, render_mode=None)
default_joint_pos = _probe_env8.scene["robot"].data.default_joint_pos[0].cpu().numpy()
left_hip_default_deg = np.degrees(default_joint_pos[JOINT_NAMES.index("left_hip_pitch_joint")])
print(f"left_hip_pitch_joint の、デフォルト角度: {left_hip_default_deg:.2f} 度")
_probe_env8.close()
del _probe_env8
torch.cuda.empty_cache()

# 転倒した各seedについて、転倒直前(最後の10ステップ)の、左hip_pitch実測角度の平均を、
# デフォルト角度と比較する
print()
print("=== 転倒した各seedの、直前(最後の10ステップ)の、左hip_pitch実測角度 ===")
fell_count = 0
for seed, df in mechanism_logs_left_hip.items():
    fell = len(df) < 250
    if not fell:
        print(f"  seed{seed}: 転倒せず(250ステップ完走)。対象外。")
        continue
    fell_count += 1
    tail = df.tail(10)
    col = "jpos_left_hip_pitch_joint_deg"
    terminal_val = tail[col].iloc[-1]
    diff_from_default = terminal_val - left_hip_default_deg
    print(f"  seed{seed}: 転倒直前の実測角度 = {terminal_val:.2f}度  "
          f"(デフォルト角度との差: {diff_from_default:+.2f}度)、n_steps={len(df)}")

print()
print(f"{fell_count}個のseedで、転倒直前の実測角度を、デフォルト角度と比較しました。")
print("差が全て±0.00度前後であれば、right_hip_pitchと同じ、bilateralな機序が確定します。")


## 16. 追加実験5: 用量反応実験(段階的な出力減衰)

これまでの実験は、すべて「出力を完全にゼロにする」ablationだった。ここでは、
**出力を、100%(正常)から0%(完全ablation)まで、段階的に減衰**させ、以下を検証する。

1. **「デフォルト姿勢への、急激な収束」は、"完全にゼロにした時だけ"起こる現象か、
   それとも、部分的な減衰でも、なだらかに起こり始めるのか**
2. **これまで確認してきた「左右差」は、完全ablationだけでなく、部分的な減衰でも
   一貫して見られるか**

対象: `right/left ankle_pitch`・`right/left hip_pitch`(§3.1・§3.2で確認済みの、
4つの重要関節)。減衰レベル: 100%・80%・60%・40%・20%・0%(6段階)。速度は0.5m/s固定
(スクリーニングと同じ条件)。8seed全部。**4関節×6レベル×8seed=192本**。

**resume可能な設計**(§6・§13と同じ考え方): 1本終わるごとに、即座にCSVへ追記し、
既存のCSVがあれば、完了済みの組み合わせを自動的にスキップする。


In [ ]:
def run_graded_ablation_rollout(checkpoint_path, ablate_joint_idx, degradation_level,
                                   forward_speed_mps, steps=250, env_seed=ENV_SEED):
    # run_ablation_rolloutと同じ構造だが、出力を完全にゼロにするのではなく、
    # degradation_level(残す割合、1.0=正常、0.0=完全ablation)倍する
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    with torch.no_grad():
        final_layer = runner.alg.actor.mlp[-1]
        final_layer.weight[ablate_joint_idx, :] *= degradation_level
        final_layer.bias[ablate_joint_idx] *= degradation_level

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {"fell_over": fell_over, "n_steps_completed": n_steps_completed}


In [ ]:
DOSE_RESPONSE_CSV = "g1_dose_response.csv"
DOSE_RESPONSE_JOINTS = [
    "right_ankle_pitch_joint", "left_ankle_pitch_joint",
    "right_hip_pitch_joint", "left_hip_pitch_joint",
]
DOSE_RESPONSE_LEVELS = [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]
DOSE_RESPONSE_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
DOSE_RESPONSE_SPEED = 0.5

# --- resumeのための準備 ---
if os.path.exists(DOSE_RESPONSE_CSV):
    _existing_dr_df = pd.read_csv(DOSE_RESPONSE_CSV)
    dr_completed = set(zip(_existing_dr_df["joint_name"], _existing_dr_df["seed"],
                            _existing_dr_df["degradation_level"]))
    print(f"既存のログを検出: {len(_existing_dr_df)}行、{len(dr_completed)}組が完了済み")
else:
    dr_completed = set()
    print("新規に開始します。")

total_dr_combos = len(DOSE_RESPONSE_JOINTS) * len(DOSE_RESPONSE_LEVELS) * len(DOSE_RESPONSE_SEEDS)
n_done = 0

for joint_name in DOSE_RESPONSE_JOINTS:
    joint_idx = JOINT_NAMES.index(joint_name)
    for level in DOSE_RESPONSE_LEVELS:
        for seed in DOSE_RESPONSE_SEEDS:
            if (joint_name, seed, level) in dr_completed:
                continue
            run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
            if not run_dirs:
                print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
                continue
            ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                       key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

            n_done += 1
            print(f"=== [{len(dr_completed)+n_done}/{total_dr_combos}] "
                  f"{joint_name} / level={level} / seed{seed} ===")
            result = run_graded_ablation_rollout(ckpt, joint_idx, level, DOSE_RESPONSE_SPEED)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "joint_name": joint_name, "joint_idx": joint_idx, "seed": seed,
                "degradation_level": level, "speed_mps": DOSE_RESPONSE_SPEED, **result
            }])
            write_header = not os.path.exists(DOSE_RESPONSE_CSV)
            row_df.to_csv(DOSE_RESPONSE_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{DOSE_RESPONSE_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [ ]:
dose_response_df = pd.read_csv(DOSE_RESPONSE_CSV)

# 関節 x 減衰レベル ごとの、8seed中の転倒率を集計する
dr_pivot = (
    dose_response_df.groupby(["joint_name", "degradation_level"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
dr_pivot["fall_rate"] = dr_pivot["n_falls"] / dr_pivot["n_seeds"]

print("=== 関節 x 残存率 ごとの転倒率 ===")
dr_table = dr_pivot.pivot(index="degradation_level", columns="joint_name", values="fall_rate")
dr_table = dr_table.sort_index(ascending=False)  # 100%から0%の順に表示
print(dr_table.to_string())
dr_table.to_csv("g1_dose_response_summary_table.csv")

print()
print("=== 左右差が、部分的な減衰でも持続するか ===")
for pair_base in ["ankle_pitch", "hip_pitch"]:
    right_name = f"right_{pair_base}_joint"
    left_name = f"left_{pair_base}_joint"
    for level in DOSE_RESPONSE_LEVELS:
        r = dr_pivot[(dr_pivot["joint_name"] == right_name) & (dr_pivot["degradation_level"] == level)]
        l = dr_pivot[(dr_pivot["joint_name"] == left_name) & (dr_pivot["degradation_level"] == level)]
        r_rate = r["fall_rate"].values[0] if len(r) else float("nan")
        l_rate = l["fall_rate"].values[0] if len(l) else float("nan")
        print(f"  {pair_base}, 残存率{level*100:.0f}%: 右={r_rate:.3f}, 左={l_rate:.3f}")


=== 関節 x 残存率 ごとの転倒率 ===
joint_name         left_ankle_pitch_joint  left_hip_pitch_joint  right_ankle_pitch_joint  right_hip_pitch_joint
degradation_level                                                                                              
1.0                                 0.000                 0.000                     0.00                  0.000
0.8                                 0.000                 0.000                     0.00                  0.000
0.6                                 0.000                 0.000                     0.00                  0.000
0.4                                 0.000                 0.000                     0.25                  0.000
0.2                                 0.375                 0.000                     1.00                  0.000
0.0                                 0.875                 0.125                     1.00                  0.125

=== 左右差が、部分的な減衰でも持続するか ===
  ankle_pitch, 残存率100%: 右=0.000, 左=0.000
  ankle_pit